# TrustHire — train video deepfake detectors on Kaggle/Colab

Train on **FaceForensics++ c23**, test in-domain and **cross-dataset on Celeb-DF v2** (the standard generalisation protocol). Add the datasets via Kaggle *Add data* (your own private upload of the officially obtained data) or mount Drive. Nothing here downloads gated data on your behalf.

In [ ]:
# --- environment (Kaggle: enable GPU T4x2 / P100; Colab: Runtime > GPU) ---
!pip -q install uv
!git clone https://github.com/Navyab27/TrustHire.git trusthire || true
%cd trusthire
!uv pip install --system -e . 2>&1 | tail -2
!python -m trusthire.cli download-models --profile gpu

In [ ]:
FFPP = '/kaggle/input/ffpp-c23'      # official FaceForensics++ layout
CELEB = '/kaggle/input/celeb-df-v2'  # Celeb-DF v2 layout
!python -m trusthire.video.prepare manifest --ffpp $FFPP --celebdf $CELEB -o videos.csv
!python -m trusthire.video.prepare crops videos.csv -o /kaggle/working/faces --frames 32

## 1. Baselines (no training): DeepfakeBench EfficientNet-B4 / Xception

In [ ]:
!python -m trusthire.video.eval --crops /kaggle/working/faces/crops.csv --backbones effnb4 xception --test-datasets ffpp celebdf --conditions clean jpeg_q30 lowres_x3

## 2. Modern backbones, fine-tuned with heavy augmentation
ConvNeXt-V2 and DINOv2 are candidates to beat the FF++-only baselines on Celeb-DF; keep whichever wins.

In [ ]:
for bb in ['convnextv2_tiny.fcmae_ft_in22k_in1k', 'vit_small_patch14_dinov2.lvd142m']:
    !python -m trusthire.video.train stage=frame backbone={bb} data.crops=/kaggle/working/faces/crops.csv epochs=8 batch_size=64 compile=true
!python -m trusthire.video.eval --crops /kaggle/working/faces/crops.csv --backbones convnextv2_tiny.fcmae_ft_in22k_in1k vit_small_patch14_dinov2.lvd142m --test-datasets ffpp celebdf

## 3. Temporal heads (BiLSTM vs transformer) on the winning backbone

In [ ]:
for t in ['bilstm', 'transformer']:
    !python -m trusthire.video.train stage=temporal backbone=effnb4 temporal={t} data.crops=/kaggle/working/faces/crops.csv

## 4. Export
Download `~/.trusthire/models/video/*_trained.pth`, `temporal_*.pt` and `results/video/benchmark.*` and copy them into the same paths on the demo machine.

In [ ]:
!ls -la ~/.trusthire/models/video results/video && cat results/video/benchmark.md